# A.1 Aim: Encoder-Decoder Architecture with Attention
- Implement encoder decoder architecture with attention on the given dataset
- Compare the performance of the above model with simple encoder-decoder architecture
# A.5 Task to be completed
- Design, implement and train an English to Hindi Machine Translation System for the given dataset using Encoder-Decoder Architecture with attention
- Compare the results of the above model with the simple encoder-decoder model done in Lab7. Keep the dataset size, batch size, number of epochs and all other relevant parameters the same for both the models.


In [1]:
# Installing and Importing necessary libraries
# !pip install datasets gradio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from datasets import load_dataset
import gradio as gr
import random
import time
import math
import numpy as np

# Set random seed for reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [2]:
# 1. Dataset Loading and Preprocessing
print("Loading a subset of cfilt/iitb-english-hindi dataset...")

# Load more data to compensate for filtering
dataset = load_dataset("cfilt/iitb-english-hindi", split="train[:50000]")

# Filter out noisy software localization strings
def is_clean(ex):
    en = ex['translation']['en']
    hi = ex['translation']['hi']
    return (
        '%s' not in en and '%s' not in hi and
        '%d' not in en and '%d' not in hi and
        len(en.split()) >= 3 and len(hi.split()) <= 30
    )

dataset = dataset.filter(is_clean)
print(f"Filtered dataset size: {len(dataset)}")

# Simple whitespace-based tokenization and vocab building to keep dependencies minimal
class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

en_vocab = Vocab("english")
hi_vocab = Vocab("hindi")

en_sentences = []
hi_sentences = []

for ex in dataset:
    en_sent = ex['translation']['en'].lower()
    hi_sent = ex['translation']['hi']
    en_vocab.add_sentence(en_sent)
    hi_vocab.add_sentence(hi_sent)
    en_sentences.append(en_sent)
    hi_sentences.append(hi_sent)

print(f"English Vocab Size: {en_vocab.n_words}")
print(f"Hindi Vocab Size: {hi_vocab.n_words}")

# Convert sentences to tensor
def tensorFromSentence(vocab, sentence, max_len=20):
    indexes = [vocab.word2index.get(word, 3) for word in sentence.split(' ')]
    indexes = indexes[:max_len-2]
    indexes.insert(0, 1) # <sos>
    indexes.append(2)    # <eos>
    # padding
    indexes = indexes + [0] * (max_len - len(indexes))
    return torch.tensor(indexes, dtype=torch.long, device=device)

from torch.utils.data import TensorDataset, DataLoader

MAX_LEN = 20
BATCH_SIZE = 64

src_tensors = torch.stack([tensorFromSentence(en_vocab, s, MAX_LEN) for s in en_sentences])
trg_tensors = torch.stack([tensorFromSentence(hi_vocab, s, MAX_LEN) for s in hi_sentences])

train_data = TensorDataset(src_tensors, trg_tensors)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)


Loading a subset of cfilt/iitb-english-hindi dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filtered dataset size: 29916
English Vocab Size: 4352
Hindi Vocab Size: 4558


In [3]:
# 2. Simple Encoder-Decoder Architecture
class SimpleEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class SimpleDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class SimpleSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
            
        return outputs


In [4]:
# 3. Encoder-Decoder with Attention
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.attention = Attention(hid_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(hid_dim * 2 + emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        
        a = self.attention(hidden[-1], encoder_outputs)
        a = a.unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)
        
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        weighted = weighted.squeeze(1)
        
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden, cell

class AttnEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))
        
        hidden = hidden.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.n_layers, 1, 1)
        
        return outputs, hidden, cell

class AttnSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        
        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
            
        return outputs


In [5]:
# 4. Training Initialization & Loop
# Matching parameters equivalent to the simple Seq2Seq model from Lab7 (EXP6)
INPUT_DIM = en_vocab.n_words
OUTPUT_DIM = hi_vocab.n_words
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

EPOCHS = 5
CLIP = 1

# Instantiate Models
simple_enc = SimpleEncoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
simple_dec = SimpleDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
simple_model = SimpleSeq2Seq(simple_enc, simple_dec, device).to(device)

attn_enc = AttnEncoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
attn_dec = AttentionDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
attn_model = AttnSeq2Seq(attn_enc, attn_dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)
            
simple_model.apply(init_weights)
attn_model.apply(init_weights)

optimizer_simple = optim.Adam(simple_model.parameters())
optimizer_attn = optim.Adam(attn_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0) # Index 0 refers to <pad>

def train_model(model, optimizer, dataloader, model_name):
    model.train()
    print(f"\n--- Training {model_name} ---")
    for epoch in range(EPOCHS):
        epoch_loss = 0
        for i, (src, trg) in enumerate(dataloader):
            src, trg = src.to(device), trg.to(device)
            optimizer.zero_grad()
            output = model(src, trg)
            
            output_dim = output.shape[-1]
            output = output[:, 1:, :].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            
            loss = criterion(output, trg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optimizer.step()
            epoch_loss += loss.item()
            
        print(f"Epoch: {epoch+1:02} | Loss: {epoch_loss/len(dataloader):.3f}")

# Train Both
print("Starting Training...")
train_model(simple_model, optimizer_simple, train_loader, "Simple Encoder-Decoder")
train_model(attn_model, optimizer_attn, train_loader, "Encoder-Decoder with Attention")


Starting Training...

--- Training Simple Encoder-Decoder ---
Epoch: 01 | Loss: 5.832
Epoch: 02 | Loss: 4.948
Epoch: 03 | Loss: 4.398
Epoch: 04 | Loss: 3.890
Epoch: 05 | Loss: 3.411

--- Training Encoder-Decoder with Attention ---
Epoch: 01 | Loss: 5.005
Epoch: 02 | Loss: 2.762
Epoch: 03 | Loss: 1.325
Epoch: 04 | Loss: 0.729
Epoch: 05 | Loss: 0.476


In [6]:
# 5. Intuitive Dashboard using Gradio
def translate_sentence(sentence, model, is_attn=False):
    model.eval()
    tokens = sentence.lower().split(' ')
    indexes = [en_vocab.word2index.get(word, 3) for word in tokens]
    indexes.insert(0, 1)
    indexes.append(2)
    src_tensor = torch.tensor(indexes, dtype=torch.long, device=device).unsqueeze(0)
    
    with torch.no_grad():
        if is_attn:
            encoder_outputs, hidden, cell = model.encoder(src_tensor)
        else:
            hidden, cell = model.encoder(src_tensor)
            
    trg_indexes = [1]
    for i in range(MAX_LEN):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long, device=device)
        
        with torch.no_grad():
            if is_attn:
                output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs)
            else:
                output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
                
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == 2:
            break
            
    trg_tokens = [hi_vocab.index2word.get(i, '<unk>') for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])

def translation_dashboard(text):
    simple_result = translate_sentence(text, simple_model, is_attn=False)
    attn_result = translate_sentence(text, attn_model, is_attn=True)
    return simple_result, attn_result

# Create Gradio Interface
interface = gr.Interface(
    fn=translation_dashboard,
    inputs=gr.Textbox(lines=2, placeholder="Enter English Text Here...", label="Input English Sentence"),
    outputs=[
        gr.Textbox(label="Simple Encoder-Decoder Output"),
        gr.Textbox(label="Attention Encoder-Decoder Output")
    ],
    title="English to Hindi Machine Translation",
    description="Comparing Simple Encoder-Decoder with Attention-based Encoder-Decoder Model."
)

# Launch the app inline in notebook
interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c180508e4d8feb41a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
